# Exp 5: Sentiment analysis

In [1]:
import json
import os
import re
import sys
import typing as ty
from collections import Counter
from compression import zstd

import jieba
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

BASE_DIR = "dataset"
SENTIMENT_DICT_PATH = f"{BASE_DIR}/BosonNLP_sentiment_score.txt.zst"
DEGREE_ADV_PATH = f"{BASE_DIR}/程度副词（中文）.txt.zst"
NEGATION_PATH = f"{BASE_DIR}/否定词.txt.zst"
STOP_WORDS_PATH = f"{BASE_DIR}/stoplist.txt.zst"
CSV_PATH = f"{BASE_DIR}/流浪地球.csv.zst"
POS_PATH = f"{BASE_DIR}/pos.txt.zst"
NEG_PATH = f"{BASE_DIR}/neg.txt.zst"

OUT_DIR = "out"
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
TEST_SENTENCES: tuple[str, ...] = (
    "电影比预期要更恢宏磅礴",
    "煽情显得太尴尬",
    # My own test sentences
    "我不懂，但我大受震撼。",
    "这么好的题材拍成这样也是没谁了，我真的会谢",
)

## 1. Dictionary-based sentiment analysis

```mermaid
flowchart LR
    A("Sentence") --> B["Segmentation"]
    B --> C["Filter stop words"]
    C --> D["Match sentiment dictionary"]
    C --> E["Match degree dictionary"]
    C --> F["Match negation dictionary"]
    D --> G["Compute sentiment score"]
    E --> G
    F --> G
    G --> H("Score")
```

### 1.1. Scoring scheme

In [3]:
def sentiment_score(
    x: ty.Iterable[str],
    sentiment_dict: ty.Mapping[str, float],
    degree_dict: ty.Mapping[str, float],
    negations: ty.Set[str],
    stop_words: ty.Set[str],
) -> float:
    score = 0.0
    w = 1.0
    for word in x:
        if word in stop_words:
            continue
        elif word in degree_dict:
            w *= degree_dict[word]
        elif word in negations:
            w = -w
        elif word in sentiment_dict:
            score += w * sentiment_dict[word]
            w = 1.0

    return score

### 1.2. Load dictionaries

In [4]:
# 超有范 6.04397724361
RE_SENTIMENT_LINE = re.compile(r"^(.+?)\s+(-?\d+(?:\.\d+)?)$")
sentiment_dict: dict[str, float] = {}
with zstd.open(SENTIMENT_DICT_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        match RE_SENTIMENT_LINE.match(line.strip()):
            case None:
                pass
            case m:
                sentiment_dict[m.group(1)] = float(m.group(2))
print(f"Loaded {len(sentiment_dict)} sentiment words")

# 百分之百 2
RE_DEGREE_LINE = re.compile(r"^(.+?)\s+(-?\d+(?:\.\d+)?)$")
degree_dict: dict[str, float] = {}
with zstd.open(DEGREE_ADV_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        match RE_DEGREE_LINE.match(line.strip()):
            case None:
                pass
            case m:
                degree_dict[m.group(1)] = float(m.group(2))
print(f"Loaded {len(degree_dict)} degree adverbs")

# 不大
negations: set[str] = set()
with zstd.open(NEGATION_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            negations.add(line)
print(f"Loaded {len(negations)} negation words")

# 啊
stop_words: set[str] = set()
with zstd.open(STOP_WORDS_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            stop_words.add(line)
print(f"Loaded {len(stop_words)} stop words")

Loaded 114764 sentiment words
Loaded 214 degree adverbs
Loaded 71 negation words
Loaded 1900 stop words


### 1.3. Evaluation

In [5]:
for s in TEST_SENTENCES:
    words: list[str] = jieba.lcut(s)
    score = sentiment_score(words, sentiment_dict, degree_dict, negations, stop_words)
    print(
        f"'{s}' -> {score:.4f}  ({'positive' if score > 1 else 'negative' if score < -1 else 'neutral'})"
    )
    for w in words:
        if w in stop_words:
            print(f"\t'{w}'\t-\t-")
        elif w in degree_dict:
            print(f"\t'{w}'\tD\t{degree_dict[w]}")
        elif w in negations:
            print(f"\t'{w}'\tN\t-1.0")
        elif w in sentiment_dict:
            print(f"\t'{w}'\tS\t{sentiment_dict[w]:.4f}")
        else:
            print(f"\t'{w}'\t?\t-")
    print()

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\MANTLE~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.498 seconds.
Prefix dict has been built successfully.


'电影比预期要更恢宏磅礴' -> 3.8912  (positive)
	'电影'	S	0.5527
	'比'	-	-
	'预期'	S	0.8613
	'要'	-	-
	'更'	-	-
	'恢宏'	S	2.1157
	'磅礴'	S	0.3615

'煽情显得太尴尬' -> -2.5782  (negative)
	'煽情'	?	-
	'显得'	S	0.6529
	'太'	D	1.5
	'尴尬'	S	-2.1540

'我不懂，但我大受震撼。' -> 2.6909  (positive)
	'我'	-	-
	'不'	-	-
	'懂'	S	0.7089
	'，'	-	-
	'但'	-	-
	'我'	-	-
	'大受'	?	-
	'震撼'	S	1.9821
	'。'	-	-

'这么好的题材拍成这样也是没谁了，我真的会谢' -> 0.8086  (neutral)
	'这么'	-	-
	'好'	-	-
	'的'	-	-
	'题材'	S	0.8387
	'拍'	S	0.8588
	'成'	S	0.3705
	'这样'	-	-
	'也'	-	-
	'是'	-	-
	'没'	-	-
	'谁'	-	-
	'了'	-	-
	'，'	-	-
	'我'	-	-
	'真的'	S	-1.2593
	'会谢'	?	-



## 2. Naive-Bayesian sentiment analysis